In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import string
import os
import pacmap
from sklearn.cluster import DBSCAN
import datetime
from sklearn.preprocessing import StandardScaler

In [2]:
# Load the CSV file containing SMILES
file_path = "../DATA/molecular_descriptors_concat_data_product2_Target-Logcmc.csv"
# \DATA\molecular_descriptors_concat_data_product2_Target-Logcmc.csv

In [3]:
df1 = pd.read_csv(file_path)

In [4]:
df1.columns

Index(['Name', 'cmc', 'TARGET', 'SMILES', 'MW', 'LogP', 'TPSA', 'HBA', 'HBD',
       'RotBonds', 'NumAromRings', 'FractionCSP3', 'NumRings',
       'HeavyAtomCount', 'CarbonCount', 'AliphaticCarbonCount', 'NumOH',
       'NumCarbonyls', 'NumCarboxyls', 'NumEthers', 'NumSulfurs', 'NumAmines',
       'LabuteASA', 'NumChargedAtoms', 'MolMR', 'LongestAliphaticChain',
       'PolarRatio', 'Chi0v', 'Chi1v'],
      dtype='object')

In [5]:
df1.describe()

,cmc,TARGET,MW,LogP,TPSA,HBA,HBD,RotBonds,NumAromRings,FractionCSP3,...,NumEthers,NumSulfurs,NumAmines,LabuteASA,NumChargedAtoms,MolMR,LongestAliphaticChain,PolarRatio,Chi0v,Chi1v
count,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,...,104.000000,104.0,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000
mean,0.720192,1.932080,1161.793746,6.034258,216.832115,20.846154,1.528846,67.730769,0.807692,0.931512,...,18.451923,0.0,0.201923,484.438962,0.173077,304.725349,12.490385,2.742959,50.425824,30.140081
std,3.552481,0.819256,1248.246911,10.198670,221.152424,22.486142,2.919492,68.573854,4.748177,0.122439,...,22.417001,0.0,0.863272,522.541568,0.565038,334.536878,11.008377,0.385003,53.792719,31.659153
min,0.000000,0.000000,258.219495,0.839900,47.920000,3.000000,0.000000,13.000000,0.000000,0.395626,...,1.000000,0.0,0.000000,111.544622,0.000000,73.523600,2.000000,1.755747,11.649764,7.676132
25%,0.030000,1.491362,520.393705,3.898050,95.970000,8.750000,1.000000,29.750000,0.000000,0.917500,...,6.000000,0.0,0.000000,217.527267,0.000000,139.295775,9.000000,2.519394,23.240619,14.155787
50%,0.070000,1.851258,718.357407,4.390600,130.990000,13.000000,1.000000,44.000000,0.000000,1.000000,...,10.000000,0.0,0.000000,309.874058,0.000000,191.551800,12.000000,2.665065,32.226150,19.210843
75%,0.312500,2.496197,1256.330160,5.488950,250.980000,25.250000,1.000000,77.250000,0.000000,1.000000,...,22.000000,0.0,0.000000,532.937032,0.000000,323.981600,13.000000,2.933463,53.733581,32.085384
max,33.700000,4.527643,7708.643760,101.686300,1284.740000,138.000000,30.000000,413.000000,48.000000,1.000000,...,137.000000,0.0,8.000000,3386.386320,2.000000,2299.580200,114.000000,3.986452,330.956950,203.819580


In [6]:
names = df1['Name']
targets = df1['TARGET']

if not 'Index' in df1.columns:
    # df['Index'] = range(df.shape[0])  # Index column
    df1['Index'] = range(1, df1.shape[0]+ 1)

In [7]:
df1.columns

Index(['Name', 'cmc', 'TARGET', 'SMILES', 'MW', 'LogP', 'TPSA', 'HBA', 'HBD',
       'RotBonds', 'NumAromRings', 'FractionCSP3', 'NumRings',
       'HeavyAtomCount', 'CarbonCount', 'AliphaticCarbonCount', 'NumOH',
       'NumCarbonyls', 'NumCarboxyls', 'NumEthers', 'NumSulfurs', 'NumAmines',
       'LabuteASA', 'NumChargedAtoms', 'MolMR', 'LongestAliphaticChain',
       'PolarRatio', 'Chi0v', 'Chi1v', 'Index'],
      dtype='object')

In [8]:
columns_to_drop = ['Name', 'TARGET', 'Index']

if 'cmc' in df1.columns:
    columns_to_drop.append('cmc')
elif 'log(cmc*1000+1)' in df1.columns:
    columns_to_drop.append('log(cmc*1000+1)')

df1.drop(columns=columns_to_drop, axis=1, inplace=True)

In [9]:
# Drop columns with SMILES data
cols_to_drop = [col for col in ['SMILES', 'smiles', 'Smiles'] if col in df1.columns]
if cols_to_drop:
    df1.drop(cols_to_drop, axis=1, inplace=True)

In [10]:
df1.columns

Index(['MW', 'LogP', 'TPSA', 'HBA', 'HBD', 'RotBonds', 'NumAromRings',
       'FractionCSP3', 'NumRings', 'HeavyAtomCount', 'CarbonCount',
       'AliphaticCarbonCount', 'NumOH', 'NumCarbonyls', 'NumCarboxyls',
       'NumEthers', 'NumSulfurs', 'NumAmines', 'LabuteASA', 'NumChargedAtoms',
       'MolMR', 'LongestAliphaticChain', 'PolarRatio', 'Chi0v', 'Chi1v'],
      dtype='object')

In [11]:
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df1)

In [12]:
df1.describe()  # Before scaling


,MW,LogP,TPSA,HBA,HBD,RotBonds,NumAromRings,FractionCSP3,NumRings,HeavyAtomCount,...,NumEthers,NumSulfurs,NumAmines,LabuteASA,NumChargedAtoms,MolMR,LongestAliphaticChain,PolarRatio,Chi0v,Chi1v
count,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,...,104.000000,104.0,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000
mean,1161.793746,6.034258,216.832115,20.846154,1.528846,67.730769,0.807692,0.931512,0.913462,80.096154,...,18.451923,0.0,0.201923,484.438962,0.173077,304.725349,12.490385,2.742959,50.425824,30.140081
std,1248.246911,10.198670,221.152424,22.486142,2.919492,68.573854,4.748177,0.122439,4.746200,87.682738,...,22.417001,0.0,0.863272,522.541568,0.565038,334.536878,11.008377,0.385003,53.792719,31.659153
min,258.219495,0.839900,47.920000,3.000000,0.000000,13.000000,0.000000,0.395626,0.000000,18.000000,...,1.000000,0.0,0.000000,111.544622,0.000000,73.523600,2.000000,1.755747,11.649764,7.676132
25%,520.393705,3.898050,95.970000,8.750000,1.000000,29.750000,0.000000,0.917500,0.000000,34.750000,...,6.000000,0.0,0.000000,217.527267,0.000000,139.295775,9.000000,2.519394,23.240619,14.155787
50%,718.357407,4.390600,130.990000,13.000000,1.000000,44.000000,0.000000,1.000000,0.000000,49.000000,...,10.000000,0.0,0.000000,309.874058,0.000000,191.551800,12.000000,2.665065,32.226150,19.210843
75%,1256.330160,5.488950,250.980000,25.250000,1.000000,77.250000,0.000000,1.000000,1.000000,86.000000,...,22.000000,0.0,0.000000,532.937032,0.000000,323.981600,13.000000,2.933463,53.733581,32.085384
max,7708.643760,101.686300,1284.740000,138.000000,30.000000,413.000000,48.000000,1.000000,48.000000,569.000000,...,137.000000,0.0,8.000000,3386.386320,2.000000,2299.580200,114.000000,3.986452,330.956950,203.819580


In [13]:
pd.DataFrame(df_scaled, columns=df1.columns).describe() 

,MW,LogP,TPSA,HBA,HBD,RotBonds,NumAromRings,FractionCSP3,NumRings,HeavyAtomCount,...,NumEthers,NumSulfurs,NumAmines,LabuteASA,NumChargedAtoms,MolMR,LongestAliphaticChain,PolarRatio,Chi0v,Chi1v
count,1.040000e+02,1.040000e+02,1.040000e+02,104.000000,1.040000e+02,1.040000e+02,104.000000,1.040000e+02,1.040000e+02,1.040000e+02,...,1.040000e+02,104.0,1.040000e+02,1.040000e+02,1.040000e+02,1.040000e+02,1.040000e+02,1.040000e+02,104.000000,1.040000e+02
mean,8.540177e-17,6.832142e-17,-1.024821e-16,0.000000,2.562053e-17,5.124106e-17,0.000000,-1.024821e-16,-8.540177e-18,1.024821e-16,...,4.270089e-18,0.0,4.270089e-17,4.270089e-17,4.270089e-17,1.024821e-16,8.540177e-18,-7.344552e-16,0.000000,1.110223e-16
std,1.004843e+00,1.004843e+00,1.004843e+00,1.004843,1.004843e+00,1.004843e+00,1.004843,1.004843e+00,1.004843e+00,1.004843e+00,...,1.004843e+00,0.0,1.004843e+00,1.004843e+00,1.004843e+00,1.004843e+00,1.004843e+00,1.004843e+00,1.004843,1.004843e+00
min,-7.273801e-01,-5.117836e-01,-7.674802e-01,-0.797495,-5.262046e-01,-8.019939e-01,-0.170930,-4.397967e+00,-1.933937e-01,-7.116208e-01,...,-7.822829e-01,0.0,-2.350370e-01,-7.170724e-01,-3.077935e-01,-6.944567e-01,-9.575604e-01,-2.576582e+00,-0.724333,-7.129923e-01
25%,-5.163290e-01,-2.104738e-01,-5.491570e-01,-0.540543,-1.820204e-01,-5.565488e-01,-0.170930,-1.149945e-01,-1.933937e-01,-5.196661e-01,...,-5.581578e-01,0.0,-2.350370e-01,-5.132687e-01,-3.077935e-01,-4.968980e-01,-3.186017e-01,-5.834961e-01,-0.507817,-5.073320e-01
50%,-3.569676e-01,-1.619444e-01,-3.900379e-01,-0.350623,-1.820204e-01,-3.477373e-01,-0.170930,5.620758e-01,-1.933937e-01,-3.563614e-01,...,-3.788577e-01,0.0,-2.350370e-01,-3.356867e-01,-3.077935e-01,-3.399374e-01,-4.476222e-02,-2.033005e-01,-0.339968,-3.468875e-01
75%,7.610211e-02,-5.372744e-02,1.551566e-01,0.196796,-1.820204e-01,1.394894e-01,-0.170930,5.620758e-01,1.832151e-02,6.765797e-02,...,1.590426e-01,0.0,-2.350370e-01,9.326134e-02,-3.077935e-01,5.783967e-02,4.651760e-02,4.972064e-01,0.061789,6.174275e-02
max,5.270235e+00,9.424293e+00,4.852216e+00,5.235277,9.799319e+00,5.059381e+00,9.987169,5.620758e-01,9.968935e+00,5.602830e+00,...,5.313921e+00,0.0,9.076905e+00,5.580418e+00,3.248931e+00,5.991911e+00,9.265779e+00,3.245463e+00,5.240294,5.512484e+00


In [15]:
file_path = "../DATA/molecular_descriptors_concat_data_product2_Target-cmc.csv"
# \DATA\molecular_descriptors_concat_data_product2_Target-Logcmc.csv
#DATA\molecular_descriptors_concat_data_product2_Target-cmc.csv

df2 = pd.read_csv(file_path)

names = df2['Name']
targets = df2['TARGET']

if not 'Index' in df2.columns:
    # df['Index'] = range(df.shape[0])  # Index column
    df2['Index'] = range(1, df1.shape[0]+ 1)


columns_to_drop = ['Name', 'TARGET', 'Index']

if 'cmc' in df2.columns:
    columns_to_drop.append('cmc')
elif 'Log(cmc*1000+1)' in df2.columns:
    columns_to_drop.append('Log(cmc*1000+1)')

df2.drop(columns=columns_to_drop, axis=1, inplace=True)

# %%
# Drop columns with SMILES data
cols_to_drop = [col for col in ['SMILES', 'smiles', 'Smiles'] if col in df2.columns]
if cols_to_drop:
    df2.drop(cols_to_drop, axis=1, inplace=True)

# %%
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df2)

# %%
print(df2.describe())  # Before scaling


# %%
pd.DataFrame(df_scaled, columns=df2.columns).describe()

                MW        LogP         TPSA         HBA         HBD  \
count   104.000000  104.000000   104.000000  104.000000  104.000000   
mean   1161.793746    6.034258   216.832115   20.846154    1.528846   
std    1248.246911   10.198670   221.152424   22.486142    2.919492   
min     258.219495    0.839900    47.920000    3.000000    0.000000   
25%     520.393705    3.898050    95.970000    8.750000    1.000000   
50%     718.357407    4.390600   130.990000   13.000000    1.000000   
75%    1256.330160    5.488950   250.980000   25.250000    1.000000   
max    7708.643760  101.686300  1284.740000  138.000000   30.000000   

         RotBonds  NumAromRings  FractionCSP3    NumRings  HeavyAtomCount  \
count  104.000000    104.000000    104.000000  104.000000      104.000000   
mean    67.730769      0.807692      0.931512    0.913462       80.096154   
std     68.573854      4.748177      0.122439    4.746200       87.682738   
min     13.000000      0.000000      0.395626    0.0

,MW,LogP,TPSA,HBA,HBD,RotBonds,NumAromRings,FractionCSP3,NumRings,HeavyAtomCount,...,NumEthers,NumSulfurs,NumAmines,LabuteASA,NumChargedAtoms,MolMR,LongestAliphaticChain,PolarRatio,Chi0v,Chi1v
count,1.040000e+02,1.040000e+02,1.040000e+02,104.000000,1.040000e+02,1.040000e+02,104.000000,1.040000e+02,1.040000e+02,1.040000e+02,...,1.040000e+02,104.0,1.040000e+02,1.040000e+02,1.040000e+02,1.040000e+02,1.040000e+02,1.040000e+02,104.000000,1.040000e+02
mean,8.540177e-17,6.832142e-17,-1.024821e-16,0.000000,2.562053e-17,5.124106e-17,0.000000,-1.024821e-16,-8.540177e-18,1.024821e-16,...,4.270089e-18,0.0,4.270089e-17,4.270089e-17,4.270089e-17,1.024821e-16,8.540177e-18,-7.344552e-16,0.000000,1.110223e-16
std,1.004843e+00,1.004843e+00,1.004843e+00,1.004843,1.004843e+00,1.004843e+00,1.004843,1.004843e+00,1.004843e+00,1.004843e+00,...,1.004843e+00,0.0,1.004843e+00,1.004843e+00,1.004843e+00,1.004843e+00,1.004843e+00,1.004843e+00,1.004843,1.004843e+00
min,-7.273801e-01,-5.117836e-01,-7.674802e-01,-0.797495,-5.262046e-01,-8.019939e-01,-0.170930,-4.397967e+00,-1.933937e-01,-7.116208e-01,...,-7.822829e-01,0.0,-2.350370e-01,-7.170724e-01,-3.077935e-01,-6.944567e-01,-9.575604e-01,-2.576582e+00,-0.724333,-7.129923e-01
25%,-5.163290e-01,-2.104738e-01,-5.491570e-01,-0.540543,-1.820204e-01,-5.565488e-01,-0.170930,-1.149945e-01,-1.933937e-01,-5.196661e-01,...,-5.581578e-01,0.0,-2.350370e-01,-5.132687e-01,-3.077935e-01,-4.968980e-01,-3.186017e-01,-5.834961e-01,-0.507817,-5.073320e-01
50%,-3.569676e-01,-1.619444e-01,-3.900379e-01,-0.350623,-1.820204e-01,-3.477373e-01,-0.170930,5.620758e-01,-1.933937e-01,-3.563614e-01,...,-3.788577e-01,0.0,-2.350370e-01,-3.356867e-01,-3.077935e-01,-3.399374e-01,-4.476222e-02,-2.033005e-01,-0.339968,-3.468875e-01
75%,7.610211e-02,-5.372744e-02,1.551566e-01,0.196796,-1.820204e-01,1.394894e-01,-0.170930,5.620758e-01,1.832151e-02,6.765797e-02,...,1.590426e-01,0.0,-2.350370e-01,9.326134e-02,-3.077935e-01,5.783967e-02,4.651760e-02,4.972064e-01,0.061789,6.174275e-02
max,5.270235e+00,9.424293e+00,4.852216e+00,5.235277,9.799319e+00,5.059381e+00,9.987169,5.620758e-01,9.968935e+00,5.602830e+00,...,5.313921e+00,0.0,9.076905e+00,5.580418e+00,3.248931e+00,5.991911e+00,9.265779e+00,3.245463e+00,5.240294,5.512484e+00
